# sequence_trend_joint_metric_reconciliation_v1

Notebook-first follow-up to `sequence_trend_joint_bc_screen_v1`.

- **Baselines**: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `sequence_trend_joint_bc_screen_v1` and prior `metric_source_reconciliation_v1`.
- **Role**: diagnose; no promotion and no actor retention.
- **Changed layer**: model-selection metric attribution only.
- **External basis**: Qlib keeps model training, backtesting, and deployment in one quant workflow; DAgger/BC warns that policies face distribution shift under their own actions; PBO/backtest-overfit literature requires explicit OOS metric discipline.
- **Hypothesis**: the latest joint BC notebook selected using `evaluate_split` relative-return fields while writing official `backtest_profile` daily paths. If calendar vs tradable-window benchmark alignment flips gates again, downstream reward/PPO work must stop until the official evaluation gate is explicit.
- **Acceptance criteria**: compare saved `evaluate_split` metrics, official daily-path calendar metrics, and tradable-window relative metrics for every candidate/fold; identify gate flips; write `selection.json` with promotion blocked.
- **Comparison-table row key**: `sequence_trend_joint_metric_reconciliation_v1`.


In [1]:

from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import compute_relative_metric_variants
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

EXP = 'sequence_trend_joint_metric_reconciliation_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXP
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'sequence_trend_joint_bc_screen_v1'
PRIOR_RECON_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'metric_source_reconciliation_v1'
CARD.mkdir(parents=True, exist_ok=True)
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}
EXTERNAL_SOURCES = [
    'https://github.com/microsoft/qlib',
    'https://arxiv.org/abs/2009.11189',
    'https://proceedings.mlr.press/v15/ross11a.html',
    'https://doi.org/10.21314/JCF.2016.322',
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')

source_selection = json.loads((SOURCE_CARD / 'selection.json').read_text(encoding='utf-8'))
prior_selection = json.loads((PRIOR_RECON_CARD / 'selection.json').read_text(encoding='utf-8'))
fold_results = pd.read_csv(SOURCE_CARD / 'bc_fold_results.csv')
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=tuple(profile.feature.packs), profile_name=profile.name)
price = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
print('source decision', source_selection.get('decision'))
print('source selected', source_selection.get('selected_candidate'))
print('lookback', LOOKBACK)


[73773:MainThread](2026-05-26 15:57:17,767) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[73773:MainThread](2026-05-26 15:57:18,469) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[73773:MainThread](2026-05-26 15:57:18,470) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[73773:MainThread](2026-05-26 15:57:21,125) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[73773:MainThread](2026-05-26 15:57:21,127) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[73773:MainThread](2026-05-26 15:57:21,129) INFO - qlib.timer - [log.py:127] - Time cost: 0.005s | Init data Done


source decision sequence_trend_joint_bc_mixed_no_promotion
source selected trend_plus_slope_low_cap15_weighted
lookback 60


In [2]:

rows = []
for result in fold_results.to_dict('records'):
    candidate = str(result['candidate'])
    fold = str(result['fold'])
    if fold not in WALK_FORWARD_WINDOWS:
        continue
    for segment in ('valid', 'test'):
        path = SOURCE_CARD / f'{candidate}_{fold}_{segment}_daily_path.csv'
        if not path.exists():
            continue
        daily = pd.read_csv(path, parse_dates=['datetime'])
        official = daily_path_metrics(daily)
        requested_seg = WALK_FORWARD_WINDOWS[fold][segment]
        effective_seg = (str(pd.Timestamp(daily['datetime'].min()).date()), str(pd.Timestamp(daily['datetime'].max()).date()))
        variants = compute_relative_metric_variants(
            strategy_return=float(official['return']),
            price_series=price,
            requested_seg=requested_seg,
            effective_seg=effective_seg,
            lookback=LOOKBACK,
            target_relative_return=TARGET_RELATIVE_RETURN,
        )
        split_relative = float(result.get(f'{segment}_relative_return', math.nan))
        split_max_dd = float(result.get(f'{segment}_max_dd', math.nan))
        split_avg_pos = float(result.get(f'{segment}_avg_pos', math.nan))
        calendar_relative = float(variants['calendar_relative_return'])
        tradable_relative = float(variants['tradable_relative_return'])
        official_max_dd = float(official['max_dd'])
        row = {
            'candidate': candidate,
            'fold': fold,
            'segment': segment,
            'daily_path': str(path.relative_to(ROOT)),
            'official_return': float(official['return']),
            'official_max_dd': official_max_dd,
            'official_avg_pos': float(official['avg_pos']),
            'split_relative_return': split_relative,
            'split_max_dd': split_max_dd,
            'split_avg_pos': split_avg_pos,
            **variants,
            'split_relative_gap_vs_calendar': float(split_relative - calendar_relative) if math.isfinite(split_relative) else math.nan,
            'split_relative_gap_vs_tradable': float(split_relative - tradable_relative) if math.isfinite(split_relative) else math.nan,
            'split_full_gate': bool(split_relative >= TARGET_RELATIVE_RETURN and split_max_dd <= TARGET_MAX_DD),
            'calendar_full_gate': bool(calendar_relative >= TARGET_RELATIVE_RETURN and official_max_dd <= TARGET_MAX_DD),
            'tradable_full_gate': bool(tradable_relative >= TARGET_RELATIVE_RETURN and official_max_dd <= TARGET_MAX_DD),
            'split_calendar_gate_flip': bool((split_relative >= TARGET_RELATIVE_RETURN) != (calendar_relative >= TARGET_RELATIVE_RETURN)),
            'calendar_tradable_gate_flip': bool((calendar_relative >= TARGET_RELATIVE_RETURN) != (tradable_relative >= TARGET_RELATIVE_RETURN)),
            'maxdd_source_gap': float(split_max_dd - official_max_dd) if math.isfinite(split_max_dd) else math.nan,
        }
        rows.append(row)

recon = pd.DataFrame(rows).sort_values(['segment', 'fold', 'candidate']).reset_index(drop=True)
recon.to_csv(CARD / 'metric_reconciliation.csv', index=False)
display(recon[[
    'candidate', 'fold', 'segment', 'official_return', 'official_max_dd',
    'split_relative_return', 'calendar_relative_return', 'tradable_relative_return',
    'split_full_gate', 'calendar_full_gate', 'tradable_full_gate',
    'split_calendar_gate_flip', 'calendar_tradable_gate_flip', 'mismatch_reason',
]])


,candidate,fold,segment,official_return,official_max_dd,split_relative_return,calendar_relative_return,tradable_relative_return,split_full_gate,calendar_full_gate,tradable_full_gate,split_calendar_gate_flip,calendar_tradable_gate_flip,mismatch_reason
0,base_plain_contract,wf_2022,test,-16.534504,18.396429,36.419124,70.053568,60.079764,False,False,False,False,False,material_warmup_benchmark_alignment_gap
1,trend_plain_contract,wf_2022,test,-16.534504,18.396429,36.419124,70.053568,60.079764,False,False,False,False,False,material_warmup_benchmark_alignment_gap
2,trend_plus_slope_low_cap15,wf_2022,test,-14.393159,16.115577,59.461195,72.194912,62.221108,False,False,False,False,False,material_warmup_benchmark_alignment_gap
3,trend_plus_slope_low_cap15_weighted,wf_2022,test,-18.860949,20.305636,61.152506,67.727122,57.753318,False,False,False,False,False,material_warmup_benchmark_alignment_gap
4,trend_plus_trend_break_any_cap10,wf_2022,test,-18.062932,19.550984,68.105462,68.525139,58.551335,False,False,False,False,False,material_warmup_benchmark_alignment_gap
5,base_plain_contract,wf_2023_2024,test,-5.811867,13.472617,30.501116,-5.455232,34.140849,False,False,True,True,True,warmup_benchmark_alignment_gate_flip
6,trend_plain_contract,wf_2023_2024,test,-1.364189,12.897887,41.557303,-1.007555,38.588527,True,False,True,True,True,warmup_benchmark_alignment_gate_flip
7,trend_plus_slope_low_cap15,wf_2023_2024,test,-14.532341,20.379351,32.530872,-14.175706,25.420375,False,False,False,True,False,material_warmup_benchmark_alignment_gap
8,trend_plus_slope_low_cap15_weighted,wf_2023_2024,test,-4.037677,12.392254,40.445822,-3.681043,35.915039,True,False,True,True,True,warmup_benchmark_alignment_gate_flip
9,trend_plus_trend_break_any_cap10,wf_2023_2024,test,-5.274448,10.998685,38.433877,-4.917813,34.678268,True,False,True,True,True,warmup_benchmark_alignment_gate_flip


In [3]:

def gate_loss(group: pd.DataFrame, relative_col: str, maxdd_col: str = 'official_max_dd') -> float:
    loss = 0.0
    for row in group.to_dict('records'):
        loss += max(0.0, TARGET_RELATIVE_RETURN - float(row[relative_col]))
        loss += 10.0 * max(0.0, float(row[maxdd_col]) - TARGET_MAX_DD)
    return float(loss)

summary_rows = []
test_recon = recon[recon['segment'].eq('test')].copy()
for candidate, group in test_recon.groupby('candidate', sort=True):
    group = group.sort_values('fold')
    row = {
        'candidate': candidate,
        'fold_count': int(len(group)),
        'split_full_gate_all': bool(group['split_full_gate'].all()),
        'calendar_full_gate_all': bool(group['calendar_full_gate'].all()),
        'tradable_full_gate_all': bool(group['tradable_full_gate'].all()),
        'split_calendar_gate_flip_any': bool(group['split_calendar_gate_flip'].any()),
        'calendar_tradable_gate_flip_any': bool(group['calendar_tradable_gate_flip'].any()),
        'split_loss': gate_loss(group, 'split_relative_return', 'split_max_dd'),
        'calendar_loss': gate_loss(group, 'calendar_relative_return', 'official_max_dd'),
        'tradable_loss': gate_loss(group, 'tradable_relative_return', 'official_max_dd'),
        'min_split_relative_return': float(group['split_relative_return'].min()),
        'min_calendar_relative_return': float(group['calendar_relative_return'].min()),
        'min_tradable_relative_return': float(group['tradable_relative_return'].min()),
        'max_official_max_dd': float(group['official_max_dd'].max()),
        'max_split_max_dd': float(group['split_max_dd'].max()),
    }
    for fold in WALK_FORWARD_WINDOWS:
        sub = group[group['fold'].eq(fold)]
        if not sub.empty:
            rec = sub.iloc[0]
            row[f'{fold}_split_relative_return'] = float(rec['split_relative_return'])
            row[f'{fold}_calendar_relative_return'] = float(rec['calendar_relative_return'])
            row[f'{fold}_tradable_relative_return'] = float(rec['tradable_relative_return'])
            row[f'{fold}_official_max_dd'] = float(rec['official_max_dd'])
            row[f'{fold}_split_max_dd'] = float(rec['split_max_dd'])
    summary_rows.append(row)

candidate_summary = pd.DataFrame(summary_rows).sort_values(
    ['calendar_full_gate_all', 'tradable_full_gate_all', 'calendar_loss', 'candidate'],
    ascending=[False, False, True, True],
).reset_index(drop=True)
candidate_summary.to_csv(CARD / 'candidate_reconciliation_summary.csv', index=False)
selected_candidate = str(source_selection.get('selected_candidate', ''))
source_selected_summary = candidate_summary[candidate_summary['candidate'].eq(selected_candidate)]
official_selected_candidate = str(candidate_summary.iloc[0]['candidate']) if not candidate_summary.empty else ''
official_selected_summary = candidate_summary.iloc[0].to_dict() if not candidate_summary.empty else {}
source_selected_row = source_selected_summary.iloc[0].to_dict() if not source_selected_summary.empty else {}

display(candidate_summary)


,candidate,fold_count,split_full_gate_all,calendar_full_gate_all,tradable_full_gate_all,split_calendar_gate_flip_any,calendar_tradable_gate_flip_any,split_loss,calendar_loss,tradable_loss,...,wf_2022_split_relative_return,wf_2022_calendar_relative_return,wf_2022_tradable_relative_return,wf_2022_official_max_dd,wf_2022_split_max_dd,wf_2023_2024_split_relative_return,wf_2023_2024_calendar_relative_return,wf_2023_2024_tradable_relative_return,wf_2023_2024_official_max_dd,wf_2023_2024_split_max_dd
0,trend_plain_contract,2,False,False,False,True,True,285.712567,64.971847,33.964292,...,36.419124,70.053568,60.079764,18.396429,43.571257,41.557303,-1.007555,38.588527,12.897887,13.337480
1,base_plain_contract,2,False,False,False,True,True,322.022259,69.419524,33.964292,...,36.419124,70.053568,60.079764,18.396429,43.571257,30.501116,-5.455232,34.140849,13.472617,18.630969
2,trend_plus_trend_break_any_cap10,2,False,False,False,True,True,31.679926,80.427654,45.509841,...,68.105462,68.525139,58.551335,19.550984,18.167993,38.433877,-4.917813,34.678268,10.998685,10.947237
3,trend_plus_slope_low_cap15_weighted,2,False,False,False,True,True,32.203794,86.737400,53.056358,...,61.152506,67.727122,57.753318,20.305636,18.220379,40.445822,-3.681043,35.915039,12.392254,12.470085
4,trend_plus_slope_low_cap15,2,False,False,False,True,False,103.621627,109.124981,69.528899,...,59.461195,72.194912,62.221108,16.115577,20.450912,32.530872,-14.175706,25.420375,20.379351,19.911251


In [4]:

gate_flip_rows = recon[recon['split_calendar_gate_flip'] | recon['calendar_tradable_gate_flip']].copy()
calendar_pass_count = int(candidate_summary['calendar_full_gate_all'].sum()) if not candidate_summary.empty else 0
tradable_pass_count = int(candidate_summary['tradable_full_gate_all'].sum()) if not candidate_summary.empty else 0
source_selected_calendar_gate = bool(source_selected_row.get('calendar_full_gate_all', False))
source_selected_tradable_gate = bool(source_selected_row.get('tradable_full_gate_all', False))
if calendar_pass_count > 0:
    decision = 'sequence_trend_joint_metric_reconciled_calendar_candidate_ready_no_promotion'
    next_mechanism = 'warm_start_official_qlib_backtest_then_fixed_split_wf2020_before_reward_ppo'
elif tradable_pass_count > 0:
    decision = 'sequence_trend_joint_metric_reconciled_tradable_only_no_promotion'
    next_mechanism = 'choose_explicit_tradable_window_gate_or_warm_start_official_qlib_backtest_before_reward_ppo'
else:
    decision = 'sequence_trend_joint_metric_mismatch_no_promotion'
    next_mechanism = 'do_not_run_reward_ppo; repair_warm_start_official_evaluation_or_redefine_metric_gate_first'

selection = {
    'experiment': EXP,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': ['sequence_trend_joint_bc_screen_v1', 'metric_source_reconciliation_v1'],
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'harness_metric_diagnostic'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'ppo_screen_allowed_next': False,
    'source_selected_candidate': selected_candidate,
    'source_selected_split_full_gate_all': bool(source_selected_row.get('split_full_gate_all', False)),
    'source_selected_calendar_full_gate_all': source_selected_calendar_gate,
    'source_selected_tradable_full_gate_all': source_selected_tradable_gate,
    'official_calendar_selected_candidate': official_selected_candidate,
    'calendar_full_gate_candidate_count': calendar_pass_count,
    'tradable_full_gate_candidate_count': tradable_pass_count,
    'gate_flip_count': int(len(gate_flip_rows)),
    'split_calendar_gate_flip_count': int(recon['split_calendar_gate_flip'].sum()),
    'calendar_tradable_gate_flip_count': int(recon['calendar_tradable_gate_flip'].sum()),
    'root_cause': 'evaluate_split_vs_official_backtest_warmup_alignment',
    'promotion_blocker': (
        'The source card selected with evaluate_split-style relative returns. Official backtest_profile daily paths show no calendar-full-gate candidate. '
        'Some rows recover under tradable-window relative metrics, which repeats the prior warmup benchmark alignment issue. Do not run reward/PPO from this card until official evaluation is warm-started or the tradable-window gate is explicitly adopted and replayed on fixed-split/wf_2020.'
    ),
    'next_mechanism': next_mechanism,
    'external_sources': EXTERNAL_SOURCES,
    'source_selection_decision': source_selection.get('decision'),
    'prior_reconciliation_decision': prior_selection.get('decision'),
    'sidecars': [
        'metric_reconciliation.csv',
        'candidate_reconciliation_summary.csv',
        'selection.json',
        'summary.json',
    ],
}
summary = {
    **selection,
    'source_selected_summary': source_selected_row,
    'official_calendar_selected_summary': official_selected_summary,
    'candidate_summary': candidate_summary.to_dict('records'),
    'gate_flip_rows': gate_flip_rows.to_dict('records'),
    'conclusion': [],
}
if source_selected_row:
    summary['conclusion'].append(
        f"Source selected `{selected_candidate}` had split gates {source_selected_row.get('split_full_gate_all')} but calendar gates {source_selected_row.get('calendar_full_gate_all')}; "
        f"min split relative {source_selected_row.get('min_split_relative_return'):.2f}pp, min calendar relative {source_selected_row.get('min_calendar_relative_return'):.2f}pp, "
        f"min tradable relative {source_selected_row.get('min_tradable_relative_return'):.2f}pp, max official MaxDD {source_selected_row.get('max_official_max_dd'):.2f}%."
    )
if official_selected_summary:
    summary['conclusion'].append(
        f"Best calendar-loss diagnostic row is `{official_selected_candidate}` with calendar_loss {official_selected_summary.get('calendar_loss'):.2f}, "
        f"min calendar relative {official_selected_summary.get('min_calendar_relative_return'):.2f}pp, max official MaxDD {official_selected_summary.get('max_official_max_dd'):.2f}%."
    )
summary['conclusion'].append('No promotion and no PPO/reward handoff from the source card until metric-source alignment is fixed.')
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


{
  "experiment": "sequence_trend_joint_metric_reconciliation_v1",
  "created_at": "2026-05-26T07:57:21.301638+00:00",
  "decision": "sequence_trend_joint_metric_mismatch_no_promotion",
  "role": "diagnose",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "sequence_trend_joint_bc_screen_v1",
    "metric_source_reconciliation_v1"
  ],
  "profile": "sparse_release_bc_ppo_screen_v1",
  "changed_layer": [
    "model_selection",
    "harness_metric_diagnostic"
  ],
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "promotion_allowed": false,
  "training_allowed_from_this_card": false,
  "ppo_screen_allowed_next": false,
  "source_selected_candidate": "trend_plus_slope_low_cap15_weighted",
  "source_selected_split_full_gate_all": false,
  "source_selected_calendar_full_gate_all": false,
  "source_selected_tradable_full_gate_all": false,
  "official_calendar_selected_candidate": "trend_plain_contract",
  "calendar_full_g

In [5]:

selection = json.loads((CARD / 'selection.json').read_text(encoding='utf-8'))
summary = json.loads((CARD / 'summary.json').read_text(encoding='utf-8'))
lines = '\n'.join(f"- {line}" for line in summary['conclusion'])
md = f"""
## Runtime Conclusion

Decision: `{selection['decision']}`. Promotion remains blocked.

Source selected candidate: `{selection['source_selected_candidate']}`.
Official calendar selected diagnostic row: `{selection['official_calendar_selected_candidate']}`.

{lines}

Next action: `{selection['next_mechanism']}`.
""".strip()
display(Markdown(md))
candidate_summary


## Runtime Conclusion

Decision: `sequence_trend_joint_metric_mismatch_no_promotion`. Promotion remains blocked.

Source selected candidate: `trend_plus_slope_low_cap15_weighted`.
Official calendar selected diagnostic row: `trend_plain_contract`.

- Source selected `trend_plus_slope_low_cap15_weighted` had split gates False but calendar gates False; min split relative 40.45pp, min calendar relative -3.68pp, min tradable relative 35.92pp, max official MaxDD 20.31%.
- Best calendar-loss diagnostic row is `trend_plain_contract` with calendar_loss 64.97, min calendar relative -1.01pp, max official MaxDD 18.40%.
- No promotion and no PPO/reward handoff from the source card until metric-source alignment is fixed.

Next action: `do_not_run_reward_ppo; repair_warm_start_official_evaluation_or_redefine_metric_gate_first`.

,candidate,fold_count,split_full_gate_all,calendar_full_gate_all,tradable_full_gate_all,split_calendar_gate_flip_any,calendar_tradable_gate_flip_any,split_loss,calendar_loss,tradable_loss,...,wf_2022_split_relative_return,wf_2022_calendar_relative_return,wf_2022_tradable_relative_return,wf_2022_official_max_dd,wf_2022_split_max_dd,wf_2023_2024_split_relative_return,wf_2023_2024_calendar_relative_return,wf_2023_2024_tradable_relative_return,wf_2023_2024_official_max_dd,wf_2023_2024_split_max_dd
0,trend_plain_contract,2,False,False,False,True,True,285.712567,64.971847,33.964292,...,36.419124,70.053568,60.079764,18.396429,43.571257,41.557303,-1.007555,38.588527,12.897887,13.337480
1,base_plain_contract,2,False,False,False,True,True,322.022259,69.419524,33.964292,...,36.419124,70.053568,60.079764,18.396429,43.571257,30.501116,-5.455232,34.140849,13.472617,18.630969
2,trend_plus_trend_break_any_cap10,2,False,False,False,True,True,31.679926,80.427654,45.509841,...,68.105462,68.525139,58.551335,19.550984,18.167993,38.433877,-4.917813,34.678268,10.998685,10.947237
3,trend_plus_slope_low_cap15_weighted,2,False,False,False,True,True,32.203794,86.737400,53.056358,...,61.152506,67.727122,57.753318,20.305636,18.220379,40.445822,-3.681043,35.915039,12.392254,12.470085
4,trend_plus_slope_low_cap15,2,False,False,False,True,False,103.621627,109.124981,69.528899,...,59.461195,72.194912,62.221108,16.115577,20.450912,32.530872,-14.175706,25.420375,20.379351,19.911251
